# Hipóteses de NPS - Safira

Implementação computacional das hipóteses descritas na seção 4.2.3 (`Hipóteses`) de
`documents/documentacao.md`:

- **Hipótese 1**: uma tripulação mal avaliada pode pesar tanto quanto um atraso grave
- **Hipótese 2**: o canal de compra revela um perfil de cliente com sensibilidades diferentes
- **Hipótese 4**: o detrator crônico

A base utilizada é `AMOSTRA_NPS_INTELI_FINAL`, conforme o dicionário de dados da seção 4.1.3.
Os dados não são versionados neste repositório (ver `.gitignore` e a Política de Privacidade
da seção 4.1.8). Aponte `DATA_PATH`, abaixo, para o arquivo local ou para o caminho no Google
Drive quando este notebook for aberto no Colab.

**Aviso sobre nomes de coluna:** o texto das hipóteses cita variáveis (atraso na chegada,
canal de compra, histórico do cliente) que não aparecem no dicionário de dados publicado na
seção 4.1.3, o qual documenta apenas o atraso na partida (`ESTATISTICA_ATRASOSAIDA`) e nenhum
campo de canal de compra ou de identificação do cliente ao longo do tempo
(`CLIENTE_RECORDLOCATOR` é por reserva, não por cliente). A célula de configuração abaixo
isola esses nomes em `COL` para que sejam ajustados rapidamente para os nomes reais da base
entregue pela Azul, e cada hipótese verifica a presença das colunas antes de rodar, avisando
o que falta em vez de falhar silenciosamente.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## Configuração

Ajuste `DATA_PATH` e o dicionário `COL` antes de rodar.

In [ ]:
# Caminho da base. No Colab, monte o Drive e aponte para o arquivo, por exemplo:
#   from google.colab import drive
#   drive.mount("/content/drive")
#   DATA_PATH = "/content/drive/MyDrive/.../AMOSTRA_NPS_INTELI_FINAL.csv"
DATA_PATH = "AMOSTRA_NPS_INTELI_FINAL.csv"

# Nomes de coluna. Os marcados como "não documentado" não aparecem no dicionário de dados
# da seção 4.1.3 e precisam ser conferidos contra a base real antes de rodar as hipóteses
# que dependem deles.
COL = {
    "target": "NPS_PRINCIPAL",                  # -100 Detrator, 0 Neutro, 100 Promotor
    "comissarios": "NPS_COMISSARIOS",
    "pilotos": "NPS_PILOTOS",
    "atraso_saida": "ESTATISTICA_ATRASOSAIDA",  # documentado: atraso na PARTIDA, em minutos
    "atraso_chegada": "ATRASO_CHEGADA",         # citado na hipótese 1 e no Anexo A.1; não documentado na seção 4.1.3
    "cancelamento": "CANCELAMENTO_VOO",
    "canal_compra": "CANAL_COMPRA",             # citado na hipótese 2 (Callcenter/Balcão/Site/App/Agência); não documentado na seção 4.1.3
    "wifi": "NPS_WIFI",
    "entretenimento": "NPS_ENTRETENIMENTO",
    "embarque": "NPS_EMBARQUE",
    "cliente_id": "CLIENTE_ID",                 # citado na hipótese 4 (histórico do cliente); não documentado na seção 4.1.3
    "data_voo": "DATA_STD",
}

LIMIAR_NOTA_NEGATIVA = 6   # notas 0 a 6 na escala 0-10, mesmo corte de Detrator do NPS_PRINCIPAL
LIMIAR_NOTA_POSITIVA = 9   # notas 9 e 10, mesmo corte de Promotor do NPS_PRINCIPAL
LIMIAR_ATRASO_GRAVE_MIN = 120
CANAIS_DIGITAIS = ["Site", "App", "Aplicativo", "Agencia"]  # ajustar aos valores reais de COL["canal_compra"]

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"{len(df):,} registros carregados de {DATA_PATH}")

def checar_colunas(colunas_necessarias, contexto):
    faltantes = [c for c in colunas_necessarias if c not in df.columns]
    if faltantes:
        print(f"[{contexto}] colunas ausentes na base carregada: {faltantes}")
        print(f"[{contexto}] ajuste o dicionário COL para os nomes reais antes de continuar.")
    return not faltantes

nome_atraso_chegada = COL["atraso_chegada"]
nome_atraso_saida = COL["atraso_saida"]
ATRASO_CHEGADA_DISPONIVEL = nome_atraso_chegada in df.columns
COL_ATRASO_GRAVE = nome_atraso_chegada if ATRASO_CHEGADA_DISPONIVEL else nome_atraso_saida
if not ATRASO_CHEGADA_DISPONIVEL:
    print(f"'{nome_atraso_chegada}' não encontrada, usando '{nome_atraso_saida}' "
          "(atraso na partida, documentado na seção 4.1.3) como proxy operacional.")

In [ ]:
VALORES_VERDADEIROS = {"true", "1", "sim", "yes", "y", "verdadeiro"}
VALORES_FALSOS = {"false", "0", "nao", "não", "no", "n", "falso"}

def to_bool(serie):
    # CANCELAMENTO_VOO e documentado apenas como "Booleano" na secao 4.1.3, sem
    # especificar a codificacao literal. serie.astype(bool) direto e uma armadilha aqui:
    # se a coluna vier como texto ("Sim"/"Nao"), qualquer string nao vazia e truthy em
    # Python, entao "Nao" tambem viraria True. Tratamos os formatos mais comuns e avisamos
    # se sobrar algum valor nao reconhecido em vez de deixar ele virar True silenciosamente.
    if pd.api.types.is_bool_dtype(serie):
        return serie.fillna(False)
    if pd.api.types.is_numeric_dtype(serie):
        return serie.fillna(0).astype(bool)
    normalizada = serie.astype(str).str.strip().str.lower()
    convertida = normalizada.map(lambda v: True if v in VALORES_VERDADEIROS else (False if v in VALORES_FALSOS else None))
    nao_reconhecidos = sorted(set(normalizada[convertida.isna() & serie.notna()]))
    if nao_reconhecidos:
        print(f"[to_bool] valores nao reconhecidos em '{serie.name}', tratados como False: {nao_reconhecidos}")
    return convertida.fillna(False).astype(bool)

def is_detrator(serie_nps):
    return serie_nps.eq(-100)

def voos_pontuais_sem_cancelamento(base):
    pontual = base[COL["atraso_saida"]] <= 0
    sem_cancelamento = ~to_bool(base[COL["cancelamento"]])
    return base[pontual & sem_cancelamento].copy()

def taxa_detratores(base, coluna_grupo=None):
    alvo = is_detrator(base[COL["target"]])
    if coluna_grupo is None:
        return alvo.mean()
    return base.assign(_detrator=alvo).groupby(coluna_grupo)["_detrator"].mean()

## Hipótese 1: uma tripulação mal avaliada pode pesar tanto quanto um atraso grave

Isola voos pontuais e sem cancelamento, compara a taxa de detratores entre avaliações
negativas e positivas de comissários e pilotos, e contrasta com a taxa de detratores no
pior cenário puramente operacional da base (atraso na chegada acima de 120 minutos).

In [ ]:
if checar_colunas(
    [COL["comissarios"], COL["pilotos"], COL["atraso_saida"], COL["cancelamento"], COL["target"]],
    "Hipótese 1",
):
    base_limpa = voos_pontuais_sem_cancelamento(df)
    print(f"Voos pontuais e sem cancelamento: {len(base_limpa):,} de {len(df):,}")

    def taxa_por_avaliacao(base, coluna_nota):
        notas = base[coluna_nota].dropna()
        negativa = base.loc[notas[notas <= LIMIAR_NOTA_NEGATIVA].index]
        positiva = base.loc[notas[notas >= LIMIAR_NOTA_POSITIVA].index]
        return taxa_detratores(negativa), taxa_detratores(positiva), len(negativa), len(positiva)

    for rotulo, coluna in [("Comissários", COL["comissarios"]), ("Pilotos", COL["pilotos"])]:
        neg, pos, n_neg, n_pos = taxa_por_avaliacao(base_limpa, coluna)
        print(f"{rotulo}: avaliação negativa {neg:.1%} (n={n_neg:,}), "
              f"avaliação positiva {pos:.1%} (n={n_pos:,})")

    if COL_ATRASO_GRAVE in df.columns:
        atraso_grave = df[df[COL_ATRASO_GRAVE] > LIMIAR_ATRASO_GRAVE_MIN]
        print(f"\nAtraso acima de {LIMIAR_ATRASO_GRAVE_MIN} min ({len(atraso_grave):,} voos): "
              f"{taxa_detratores(atraso_grave):.1%} de detratores")
    else:
        print(f"\n'{COL_ATRASO_GRAVE}' não encontrada, não foi possível calcular a taxa "
              "de detratores do pior cenário operacional.")

## Hipótese 2: o canal de compra revela um perfil de cliente com sensibilidades diferentes

Mesma base de voos pontuais e sem cancelamento. Primeiro compara a taxa de detratores por
canal de compra; depois, dentro de cada canal, mede o quanto uma avaliação negativa de
wifi/entretenimento e de comissários/embarque eleva a taxa de detratores em relação a uma
avaliação positiva.

In [ ]:
if checar_colunas(
    [COL["canal_compra"], COL["atraso_saida"], COL["cancelamento"], COL["target"]],
    "Hipótese 2",
):
    base_limpa = voos_pontuais_sem_cancelamento(df)
    taxa_por_canal = taxa_detratores(base_limpa, COL["canal_compra"]).sort_values()
    print("Taxa de detratores por canal de compra (voos pontuais e sem cancelamento):")
    print(taxa_por_canal.apply(lambda v: f"{v:.1%}"))

    def efeito_avaliacao_negativa(base, coluna_nota, canal):
        base_canal = base[base[COL["canal_compra"]] == canal]
        notas = base_canal[coluna_nota].dropna()
        negativa = base_canal.loc[notas[notas <= LIMIAR_NOTA_NEGATIVA].index]
        positiva = base_canal.loc[notas[notas >= LIMIAR_NOTA_POSITIVA].index]
        if len(negativa) == 0 or len(positiva) == 0:
            return np.nan
        return taxa_detratores(negativa) - taxa_detratores(positiva)

    for canal in taxa_por_canal.index:
        print(f"\n{canal}:")
        for rotulo, coluna in [
            ("wifi", COL["wifi"]),
            ("entretenimento", COL["entretenimento"]),
            ("comissários", COL["comissarios"]),
            ("embarque", COL["embarque"]),
        ]:
            efeito = efeito_avaliacao_negativa(base_limpa, coluna, canal)
            if pd.notna(efeito):
                print(f"  variação na detração ({rotulo} negativo vs. positivo): {efeito:+.1%}")
            else:
                print(f"  {rotulo}: dados insuficientes nesse canal")

## Hipótese 4: o detrator crônico

Requer um identificador de cliente estável entre respostas (`COL["cliente_id"]`) para
ordenar o histórico por `COL["data_voo"]` e comparar cada resposta com a resposta anterior
do mesmo cliente. Reproduz os três testes do quadro da seção 4.2.3 (concentração com
qui-quadrado 2x2, predição por taxas condicionais e resistência a controles operacionais
progressivos) além da regressão logística final.

In [ ]:
if checar_colunas(
    [COL["cliente_id"], COL["data_voo"], COL["target"], COL["atraso_saida"], COL["cancelamento"]],
    "Hipótese 4",
):
    historico = df.sort_values([COL["cliente_id"], COL["data_voo"]]).copy()
    historico["_detrator"] = is_detrator(historico[COL["target"]])
    historico["_pontual"] = (historico[COL["atraso_saida"]] <= 0) & (
        ~to_bool(historico[COL["cancelamento"]])
    )
    historico["_detrator_anterior"] = historico.groupby(COL["cliente_id"])["_detrator"].shift(1)
    historico["_pontual_anterior"] = historico.groupby(COL["cliente_id"])["_pontual"].shift(1)

    respostas_com_historico = historico[historico["_detrator_anterior"].notna()].copy()
    respostas_com_historico["_detrator_anterior"] = respostas_com_historico["_detrator_anterior"].astype(bool)

    print(
        f"Respostas com pelo menos uma resposta anterior do mesmo cliente: "
        f"{len(respostas_com_historico):,} de {len(historico):,} "
        f"({len(respostas_com_historico) / len(historico):.1%})"
    )

In [ ]:
# Teste 1: concentração, entre clientes com exatamente duas respostas
contagem_por_cliente = historico.groupby(COL["cliente_id"]).size()
clientes_com_duas = contagem_por_cliente[contagem_por_cliente == 2].index
duplas = historico[historico[COL["cliente_id"]].isin(clientes_com_duas)]

primeira = duplas.groupby(COL["cliente_id"])["_detrator"].first()
segunda = duplas.groupby(COL["cliente_id"])["_detrator"].last()
tabela_2x2 = pd.crosstab(primeira, segunda)

chi2, p_valor, gl, esperado = stats.chi2_contingency(tabela_2x2, correction=False)
esperado_df = pd.DataFrame(esperado, index=tabela_2x2.index, columns=tabela_2x2.columns)

print("Tabela observada (linha = primeira resposta, coluna = segunda resposta):")
print(tabela_2x2)
print("\nTabela esperada sob independência:")
print(esperado_df.round(1))
print(f"\nqui-quadrado = {chi2:.1f}; gl = {gl}; p = {p_valor:.3g}")

In [ ]:
# Teste 2: predição
taxa_pos = respostas_com_historico.loc[respostas_com_historico["_detrator_anterior"], "_detrator"].mean()
taxa_neg = respostas_com_historico.loc[~respostas_com_historico["_detrator_anterior"], "_detrator"].mean()

print(f"Taxa de detração | detratou na resposta anterior: {taxa_pos:.1%}")
print(f"Taxa de detração | não detratou na resposta anterior: {taxa_neg:.1%}")
print(f"Razão: {taxa_pos / taxa_neg:.1f}x")

# Teste 3: resistência a controles operacionais progressivos
def razao_detratou_vs_nao(base):
    pos = base.loc[base["_detrator_anterior"], "_detrator"].mean()
    neg = base.loc[~base["_detrator_anterior"], "_detrator"].mean()
    return pos, neg, pos / neg

voo_atual_perfeito = respostas_com_historico[respostas_com_historico["_pontual"]]
ambos_perfeitos = respostas_com_historico[
    respostas_com_historico["_pontual"] & respostas_com_historico["_pontual_anterior"].fillna(False)
]

print()
for rotulo, base in [
    ("Todas as respostas com histórico", respostas_com_historico),
    ("Voo atual pontual e sem cancelamento", voo_atual_perfeito),
    ("Voo atual e anterior pontuais e sem cancelamento", ambos_perfeitos),
]:
    pos, neg, razao = razao_detratou_vs_nao(base)
    print(f"{rotulo} (n={len(base):,}): {razao:.1f}x ({pos:.1%} vs. {neg:.1%})")

In [ ]:
# Regressão logística: detrator ~ detratou_antes + atraso + cancelamento
modelo_dados = respostas_com_historico.dropna(subset=[COL["atraso_saida"], COL["cancelamento"]]).copy()
modelo_dados["_cancelado"] = to_bool(modelo_dados[COL["cancelamento"]]).astype(int)
modelo_dados["_detrator_anterior_int"] = modelo_dados["_detrator_anterior"].astype(int)
modelo_dados["_detrator_int"] = modelo_dados["_detrator"].astype(int)

X = sm.add_constant(modelo_dados[["_detrator_anterior_int", COL["atraso_saida"], "_cancelado"]])
y = modelo_dados["_detrator_int"]
modelo_logit = sm.Logit(y, X).fit(disp=False)
print(modelo_logit.summary2().tables[1])

coef = modelo_logit.params["_detrator_anterior_int"]
ic_inf, ic_sup = modelo_logit.conf_int().loc["_detrator_anterior_int"]
p_valor_coef = modelo_logit.pvalues["_detrator_anterior_int"]

print(
    f"\nOdds ratio (detratou na resposta anterior): {np.exp(coef):.2f} "
    f"[IC 95%: {np.exp(ic_inf):.2f} - {np.exp(ic_sup):.2f}], p = {p_valor_coef:.3g}"
)